In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fixed: Extract DINOv2 attention via hooks
import torch
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

# Load DINOv2
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dinov2.eval().cuda()

transform = T.Compose([
    T.Resize((518, 518)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Hook to capture attention weights from last block
attention_store = {}

def hook_fn(module, input, output):
    # output is tuple: (attn_output, attn_weights) if we set need_weights
    pass

# DINOv2 uses sdpa by default which doesn't return weights
# We need to monkey-patch the last block's attention to store weights
last_block = dinov2.blocks[-1]
last_attn = last_block.attn

# Save original forward
_orig_forward = last_attn.forward

def patched_forward(x):
    B, N, C = x.shape
    qkv = last_attn.qkv(x).reshape(B, N, 3, last_attn.num_heads, C // last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)

    # Compute attention weights manually
    scale = (C // last_attn.num_heads) ** -0.5
    attn = (q @ k.transpose(-2, -1)) * scale
    attn = attn.softmax(dim=-1)

    # Store the attention map
    attention_store['last'] = attn.detach().cpu()

    # Continue with normal forward
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = last_attn.proj(x)
    x = last_attn.proj_drop(x)
    return x

# Patch it
last_attn.forward = patched_forward

# Find dataset
dataset_root = "/content/drive/MyDrive/Tea Leaf Dataset"
if not os.path.isdir(dataset_root):
    for alt in ["/content/drive/MyDrive/Tea_Leaf_Dataset_Final_JPG",
                "/content/drive/MyDrive/Tea Leaf Dataset_Processed_BMC"]:
        if os.path.isdir(alt):
            dataset_root = alt
            break

print(f"Using: {dataset_root}")
classes = sorted([d for d in os.listdir(dataset_root)
                  if os.path.isdir(os.path.join(dataset_root, d))])
# Skip non-image folders
classes = [c for c in classes if not c.startswith('.') and c != 'compression_report.xlsx']
print(f"Classes ({len(classes)}): {classes}")

fig, axes = plt.subplots(len(classes), 3, figsize=(12, 4 * len(classes)))
if len(classes) == 1:
    axes = axes[np.newaxis, :]

for i, cls_name in enumerate(classes):
    cls_dir = os.path.join(dataset_root, cls_name)
    imgs = [f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Pick 1 image
    img_path = os.path.join(cls_dir, imgs[len(imgs)//2])  # middle image
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).cuda()

    with torch.no_grad():
        _ = dinov2(img_tensor)

    # Extract attention from CLS token
    attn = attention_store['last']  # [1, num_heads, N+1, N+1]
    cls_attn = attn[0, :, 0, 1:]   # [num_heads, N_patches] — CLS attending to patches

    grid_size = int(cls_attn.shape[1] ** 0.5)  # 37 for 518/14
    att_maps = cls_attn.reshape(-1, grid_size, grid_size).numpy()

    mean_att = att_maps.mean(axis=0)

    # Find the "most focused" head (highest max attention)
    head_focus = att_maps.max(axis=(1, 2))
    best_head = head_focus.argmax()
    best_att = att_maps[best_head]

    # Plot
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{cls_name}', fontsize=11, fontweight='bold')
    axes[i, 0].axis('off')

    # Mean attention overlay
    att_resized = np.array(Image.fromarray(mean_att).resize(img.size, Image.BILINEAR))
    axes[i, 1].imshow(img)
    axes[i, 1].imshow(att_resized, cmap='hot', alpha=0.6)
    axes[i, 1].set_title('Mean Attention (all heads)', fontsize=10)
    axes[i, 1].axis('off')

    # Best head attention overlay
    best_resized = np.array(Image.fromarray(best_att).resize(img.size, Image.BILINEAR))
    axes[i, 2].imshow(img)
    axes[i, 2].imshow(best_resized, cmap='hot', alpha=0.6)
    axes[i, 2].set_title(f'Head {best_head} (most focused)', fontsize=10)
    axes[i, 2].axis('off')

plt.tight_layout()
save_path = '/content/drive/MyDrive/SSL_KD_Agri/tea_dinov2_attention_test.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved to {save_path}")

# Restore original forward
last_attn.forward = _orig_forward